# Microeconomía de la Incertidumbre

Notebook reconstruida con `sympy`, `numpy`, `scipy`, `pandas`, `seaborn` y `matplotlib` para reemplazar el enfoque excesivamente elemental por uno más cercano a un curso serio de microeconomía bajo riesgo.

**Objetivos**
- Derivar formalmente medidas Arrow-Pratt.
- Calcular equivalentes ciertos y primas de riesgo.
- Resolver seguro óptimo y portafolio óptimo con `scipy.optimize`.
- Visualizar cómo la aversión al riesgo modifica decisiones.


## Tabla de Contenidos

1. Utilidad esperada y medidas de Arrow-Pratt.
2. Equivalente cierto y prima de riesgo.
3. Seguro óptimo y sensibilidad a la carga actuarial.
4. Portafolio óptimo y aversión al riesgo.
5. Animación de la elección de portafolio.
6. Precios de estado y consumo contingente.
7. Ejercicios.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import sympy as sp
import seaborn as sns
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from scipy.optimize import brentq, minimize_scalar

sns.set_theme(style="whitegrid", context="talk")


def display_animation(anim):
    html = HTML(anim.to_jshtml(default_mode="loop"))
    plt.close(anim._fig)
    display(html)


def u_crra(w, gamma):
    w = np.asarray(w, dtype=float)
    if np.isclose(gamma, 1.0):
        return np.log(w)
    return (np.power(w, 1 - gamma) - 1) / (1 - gamma)


def expected_utility_binary(payoffs, probs, gamma):
    payoffs = np.asarray(payoffs, dtype=float)
    probs = np.asarray(probs, dtype=float)
    return np.sum(probs * u_crra(payoffs, gamma))


def certainty_equivalent(payoffs, probs, gamma, bracket):
    target = expected_utility_binary(payoffs, probs, gamma)
    return brentq(lambda c: u_crra(c, gamma) - target, *bracket)


def insurance_objective(coverage, wealth, loss, p_loss, loading, gamma):
    premium = (1 + loading) * p_loss * coverage
    w_good = wealth - premium
    w_bad = wealth - loss + coverage - premium
    if min(w_good, w_bad) <= 0:
        return 1e9
    return -expected_utility_binary([w_good, w_bad], [1 - p_loss, p_loss], gamma)


def optimal_coverage(wealth, loss, p_loss, loading, gamma):
    res = minimize_scalar(
        insurance_objective,
        bounds=(0, loss),
        method="bounded",
        args=(wealth, loss, p_loss, loading, gamma),
    )
    return res.x


def portfolio_objective(alpha, wealth, safe_r, risky_returns, probs, gamma):
    final_wealth = wealth * (alpha * (1 + risky_returns) + (1 - alpha) * (1 + safe_r))
    if np.min(final_wealth) <= 0:
        return 1e9
    return -expected_utility_binary(final_wealth, probs, gamma)


def optimal_share(wealth, safe_r, risky_returns, probs, gamma):
    res = minimize_scalar(
        portfolio_objective,
        bounds=(0, 1),
        method="bounded",
        args=(wealth, safe_r, risky_returns, probs, gamma),
    )
    return res.x


def portfolio_value_curve(alpha_values, wealth, safe_r, risky_returns, probs, gamma):
    alpha_values = np.asarray(alpha_values, dtype=float)[:, None]
    final_wealth = wealth * (
        alpha_values * (1 + risky_returns[None, :]) + (1 - alpha_values) * (1 + safe_r)
    )
    return np.sum(probs[None, :] * u_crra(final_wealth, gamma), axis=1)


print("Stack de micro bajo incertidumbre listo.")


## 1. Arrow-Pratt: derivación simbólica

Para una utilidad $u(w)$:
\[
A(w) = -\frac{u''(w)}{u'(w)}, \qquad R(w) = w A(w).
\]

Con `sympy` podemos derivar estas expresiones exactamente para CRRA y CARA, en lugar de solo citarlas.


In [ ]:
w, gamma_sym, a_sym = sp.symbols("w gamma a", positive=True)
u_crra_sym = (w ** (1 - gamma_sym) - 1) / (1 - gamma_sym)
u_cara_sym = -sp.exp(-a_sym * w)

A_crra = sp.simplify(-sp.diff(u_crra_sym, w, 2) / sp.diff(u_crra_sym, w))
R_crra = sp.simplify(w * A_crra)
A_cara = sp.simplify(-sp.diff(u_cara_sym, w, 2) / sp.diff(u_cara_sym, w))
R_cara = sp.simplify(w * A_cara)

display(sp.Eq(sp.Symbol("A_CRRA"), A_crra))
display(sp.Eq(sp.Symbol("R_CRRA"), R_crra))
display(sp.Eq(sp.Symbol("A_CARA"), A_cara))
display(sp.Eq(sp.Symbol("R_CARA"), R_cara))


## 2. Equivalente cierto y prima de riesgo

Considera una lotería binaria con riqueza final:
\[
W \in \{80, 140\}, \qquad P=0.5.
\]

El equivalente cierto $CE$ satisface $u(CE)=\mathbb{E}[u(W)]$ y la prima de riesgo es:
\[
\pi = \mathbb{E}[W] - CE.
\]

Aquí lo calculamos para una malla de aversiones relativas al riesgo.


In [ ]:
payoff_vec = np.array([80.0, 140.0])
prob_vec = np.array([0.5, 0.5])
gamma_grid = np.linspace(0.5, 6.0, 50)

ce_values = np.array([certainty_equivalent(payoff_vec, prob_vec, g, (1e-3, 200)) for g in gamma_grid])
premium_values = payoff_vec.mean() - ce_values

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="#fbfaf7")
sns.lineplot(x=gamma_grid, y=ce_values, ax=axes[0], color="#4c72b0", lw=3)
axes[0].axhline(payoff_vec.mean(), color="gray", ls="--")
axes[0].set_title("Equivalente cierto vs. gamma")
axes[0].set_xlabel("aversión relativa al riesgo")
axes[0].set_ylabel("CE")

sns.lineplot(x=gamma_grid, y=premium_values, ax=axes[1], color="#c44e52", lw=3)
axes[1].set_title("Prima de riesgo vs. gamma")
axes[1].set_xlabel("aversión relativa al riesgo")
axes[1].set_ylabel("prima")

fig.tight_layout()
plt.show()


## 3. Seguro óptimo y carga actuarial

Resolvemos numéricamente:

- riqueza inicial $w=100$;
- pérdida $L=40$ con probabilidad $0.25$;
- cobertura $q \in [0,L]$;
- prima $(1+\lambda)pq$.

Con `scipy.optimize.minimize_scalar` podemos trazar cómo la cobertura óptima cae a medida que la prima se encarece.


In [ ]:
loadings = np.linspace(0.0, 0.6, 35)
gamma_values = [1.2, 2.0, 4.0]

coverage_rows = []
for g in gamma_values:
    for loading in loadings:
        cov = optimal_coverage(wealth=100, loss=40, p_loss=0.25, loading=loading, gamma=g)
        coverage_rows.append({"gamma": g, "loading": loading, "coverage": cov})

coverage_df = pd.DataFrame(coverage_rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), facecolor="#fbfaf7")
sns.lineplot(data=coverage_df, x="loading", y="coverage", hue="gamma", palette="viridis", lw=3, ax=axes[0])
axes[0].set_title("Cobertura óptima vs. carga actuarial")
axes[0].set_ylabel("q*")

heat = coverage_df.pivot(index="gamma", columns="loading", values="coverage")
sns.heatmap(heat, cmap="crest", ax=axes[1])
axes[1].set_title("Mapa de calor de cobertura óptima")

fig.tight_layout()
plt.show()


## 4. Portafolio óptimo

Un activo riesgoso paga rendimientos:
\[
\tilde r \in \{0.18,\ -0.10\}, \qquad P=(0.55,\ 0.45),
\]
mientras el activo libre de riesgo rinde $r_f = 0.02$.

Para cada $\gamma$ resolvemos la fracción óptima $\alpha^*$ invertida en el activo riesgoso.


In [ ]:
risky_returns = np.array([0.18, -0.10])
probs = np.array([0.55, 0.45])
gamma_grid_port = np.linspace(0.5, 8.0, 60)
alpha_star = np.array([optimal_share(100, 0.02, risky_returns, probs, g) for g in gamma_grid_port])

fig, ax = plt.subplots(figsize=(10, 5), facecolor="#fbfaf7")
sns.lineplot(x=gamma_grid_port, y=alpha_star, color="#55a868", lw=3, ax=ax)
ax.set_title("Participación óptima en el activo riesgoso")
ax.set_xlabel("aversión relativa al riesgo")
ax.set_ylabel("alpha*")
plt.show()

print(pd.DataFrame({"gamma": gamma_grid_port[::10], "alpha_star": alpha_star[::10]}).round(3))


## 5. Animación: cómo cambia la función objetivo del portafolio al variar $\gamma$

Esta animación muestra la utilidad esperada como función de $\alpha$ para distintos niveles de aversión al riesgo. No solo cambia el óptimo: cambia también la curvatura del problema.


In [ ]:
alpha_grid = np.linspace(0, 1, 200)
gamma_anim = np.linspace(0.7, 6.0, 45)

fig, ax = plt.subplots(figsize=(10, 5), facecolor="#fbfaf7")
ax.set_xlim(0, 1)
objective_curves = [
    portfolio_value_curve(alpha_grid, 100, 0.02, risky_returns, probs, g) for g in gamma_anim
]
obj_min = min(curve.min() for curve in objective_curves)
obj_max = max(curve.max() for curve in objective_curves)
ax.set_ylim(obj_min - 0.02, obj_max + 0.02)
line, = ax.plot([], [], color="#4c72b0", lw=3)
optimum_line = ax.axvline(0, color="#c44e52", lw=2, ls="--")
title = ax.set_title("gamma = 0.7")
ax.set_xlabel("fracción en activo riesgoso")
ax.set_ylabel("utilidad esperada")

def update(frame):
    g = gamma_anim[frame]
    curve = objective_curves[frame]
    a_star = optimal_share(100, 0.02, risky_returns, probs, g)
    line.set_data(alpha_grid, curve)
    optimum_line.set_xdata([a_star, a_star])
    title.set_text(f"gamma = {g:.2f} | alpha* = {a_star:.3f}")
    return line, optimum_line, title

anim = FuncAnimation(fig, update, frames=len(gamma_anim), interval=120, blit=True)
display_animation(anim)


## 6. Precios de estado y consumo contingente

En un modelo de dos estados con utilidad logarítmica y precios de estado $(q_1, q_2)$, el consumidor elige $(c_1, c_2)$ maximizando:
\[
p_1 \log(c_1) + p_2 \log(c_2)
\]
sujeto a
\[
q_1 c_1 + q_2 c_2 = w_0.
\]

En utilidad log, el sistema se resuelve de forma limpia:
\[
c_s = \frac{p_s}{q_s} \cdot \frac{w_0}{\sum_j p_j}.
\]


In [ ]:
p_state = np.array([0.6, 0.4])
q_state = np.array([0.55, 0.35])
w0 = 100
c_star = (p_state / q_state) * w0 / p_state.sum()
autarky = np.array([100, 100])

alloc_df = pd.DataFrame(
    {
        "estado_1": [autarky[0], c_star[0]],
        "estado_2": [autarky[1], c_star[1]],
    },
    index=["sin comercio contingente", "óptimo con precios de estado"],
)
print(alloc_df.round(3))

fig, ax = plt.subplots(figsize=(9, 5), facecolor="#fbfaf7")
alloc_df.T.plot(kind="bar", ax=ax, color=["#9ecae1", "#3182bd"])
ax.set_title("Consumo contingente por estado")
ax.set_ylabel("consumo")
plt.xticks(rotation=0)
plt.show()


## 7. Ejercicios

1. Repite el cálculo del equivalente cierto para una lotería más dispersa.
2. Evalúa cómo cambia la cobertura óptima si la probabilidad de pérdida sube a 0.40.
3. Reestima $\alpha^*$ con una rentabilidad esperada menor del activo riesgoso.
4. Interpreta económicamente qué significa que $q_1 > p_1$.


In [ ]:
alt_ce = certainty_equivalent(np.array([60, 160]), np.array([0.5, 0.5]), gamma=2.0, bracket=(1e-3, 200))
alt_cov = optimal_coverage(wealth=100, loss=40, p_loss=0.40, loading=0.2, gamma=2.0)
alt_alpha = optimal_share(100, 0.02, np.array([0.12, -0.10]), probs, gamma=3.0)

print("Equivalente cierto alternativo:", round(alt_ce, 3))
print("Cobertura óptima con p=0.40:", round(alt_cov, 3))
print("Participación óptima con menor retorno esperado:", round(alt_alpha, 3))


## Referencias

- Pratt, J. (1964). *Risk Aversion in the Small and in the Large*.
- Arrow, K. J. (1965). *Aspects of the Theory of Risk-Bearing*.
- Mas-Colell, Whinston y Green. *Microeconomic Theory*.
- Gollier, C. *The Economics of Risk and Time*.
